In [ ]:
!pip install -q ultralytics roboflow yt-dlp opencv-python-headless

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="yHI5k5dujSHLcnqmmqQs")
project = rf.workspace("animesh-shastry").project("sard_yolo")
version = project.version(9)
dataset = version.download("yolov8")


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=8,
    imgsz=640,
    batch=8,
    name="survivor_detector",
    patience=5,
)

best_weights = "runs/detect/survivor_detector/weights/best.pt"
print("Best weights saved at:", best_weights)

In [ ]:
metrics = model.val(data=f"{dataset.location}/data.yaml")
print(metrics)

In [ ]:
!pip install -q -U yt-dlp
!yt-dlp -f "bv*+ba/best" --recode-video mp4 -o "test_drone_feed.mp4" "ytsearch1:aerial drone footage people search and rescue field"

INPUT_VIDEO = "test_drone_feed.mp4"

In [ ]:
INPUT_VIDEO = "16689238-hd_1920_1080_60fps.mp4"

In [ ]:
!pip install -q sahi

import cv2, csv, time
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

PERSON_CLASS_ID = 0
CONF_THRESH = 0.25
NMS_IOU = 0.3
SLICE_SIZE = 512
OVERLAP_RATIO = 0.25

detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=best_weights,
    confidence_threshold=CONF_THRESH,
    device="cuda:0",
)

cap = cv2.VideoCapture(INPUT_VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Video: {width}x{height} @ {fps:.1f}fps, {total_frames} frames")

writer = cv2.VideoWriter("annotated_output.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))
csv_file = open("detections.csv", "w", newline="")
csv_writer = csv.writer(csv_file)
csv_writer.writerow(["frame", "timestamp_sec", "confidence", "x1", "y1", "x2", "y2"])

frame_idx = 0
total_detections = 0
t0 = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    result = get_sliced_prediction(
        frame,
        detection_model,
        slice_height=SLICE_SIZE,
        slice_width=SLICE_SIZE,
        overlap_height_ratio=OVERLAP_RATIO,
        overlap_width_ratio=OVERLAP_RATIO,
        postprocess_type="NMS",
        postprocess_match_threshold=NMS_IOU,
        verbose=0,
    )

    ts = frame_idx / fps
    frame_person_count = 0
    for pred in result.object_prediction_list:
        if pred.category.id != PERSON_CLASS_ID:
            continue
        bbox = pred.bbox
        x1, y1, x2, y2 = int(bbox.minx), int(bbox.miny), int(bbox.maxx), int(bbox.maxy)
        conf = pred.score.value
        frame_person_count += 1
        csv_writer.writerow([frame_idx, f"{ts:.2f}", f"{conf:.3f}", x1, y1, x2, y2])
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(frame, f"{conf:.2f}", (x1, max(y1 - 8, 0)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)

    total_detections += frame_person_count
    cv2.putText(frame, f"Survivors this frame: {frame_person_count}  Frame {frame_idx}/{total_frames}",
                (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2, cv2.LINE_AA)
    writer.write(frame)
    frame_idx += 1
    if frame_idx % 20 == 0:
        print(f"Processed {frame_idx}/{total_frames} frames ({time.time()-t0:.1f}s)")

cap.release(); writer.release(); csv_file.close()
print(f"Done. {frame_idx} frames processed in {time.time()-t0:.1f}s.")
print(f"Total person detections across all frames: {total_detections}")
print("Outputs: annotated_output.mp4, detections.csv")

In [ ]:
from IPython.display import HTML
from base64 import b64encode

!ffmpeg -y -i annotated_output.mp4 -vcodec libx264 -crf 28 annotated_output_web.mp4 -loglevel quiet
mp4 = open("annotated_output_web.mp4", "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(f"<video width=640 controls><source src='{data_url}' type='video/mp4'></video>")

In [ ]:
from google.colab import files

files.download("annotated_output.mp4")
files.download("detections.csv")
files.download(best_weights)